<a href="https://colab.research.google.com/github/namproong/HDI-knowledge-graph/blob/main/NPASS/NPASS_collection_of_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

noting ข้างล่างจะเป็นการเปิดไฟล์และตัดตารางไฟล์เพื่อเอาข้อมูลสำคัญออกมา

In [ ]:
import pandas as pd

### Loading `/content/NPASS3.0_naturalproducts_generalinfo.txt`

natural prodruct เอาข้อมูลแค่ np id ,pref name ,iupac name

In [ ]:
df_generalinfo = pd.read_csv('/content/NPASS3.0_naturalproducts_generalinfo.txt', sep='\t')
display(df_generalinfo.head())

In [ ]:
df_natural_products_selected = df_generalinfo[['np_id', 'pref_name', 'iupac_name']]

display(df_natural_products_selected.head())

df_natural_products_selected.to_csv('natural_products_selected.csv', index=False)
print('File natural_products_selected.csv has been saved.')

### Loading `/content/NPASS3.0_naturalproducts_species_pair.txt`

naturalproduct species pair เอา org ID,np id

In [ ]:
df_species_pair = pd.read_csv('/content/NPASS3.0_naturalproducts_species_pair.txt', sep='\t')
display(df_species_pair.head())

In [ ]:
df_species_pair_selected = df_species_pair[['org_id', 'np_id']]
display(df_species_pair_selected.head())

In [ ]:
df_species_pair_selected.to_csv('species_pair_selected.csv', index=False)
print('File species_pair_selected.csv has been saved.')

### Loading `/content/NPASS3.0_naturalproducts_structure.txt`

อาเชื่อมฐานเคมี chembl เพื่อเอาไอดีที่แมชกันแล้วเก็บไว้ในตาราง และตารางสุดท้ายเอาเพียงแค่ np id , standard inchi ,standard inchi key,chembl id

In [ ]:
df_structure = pd.read_csv('/content/NPASS3.0_naturalproducts_structure.txt', sep='\t')
display(df_structure.head())

In [ ]:
!pip install chembl_downloader
import pandas as pd
import sqlite3

# 1. Load NPASS Natural Products Structure Data
try:
    # Assuming the file is now consistently in /content/sample_data/
    npass_structure_df = pd.read_csv(
        "/content/sample_data/NPASS3.0_naturalproducts_structure.txt",
        sep="\t",
        header=0
    )
    print("NPASS Natural Products Structure data loaded successfully.")
    print(f"Shape: {npass_structure_df.shape}")
    print("First 5 rows of NPASS data:")
    display(npass_structure_df.head())
except FileNotFoundError:
    print("Error: NPASS3.0_naturalproducts_structure.txt not found. Please ensure it's in the correct path.")
    npass_structure_df = pd.DataFrame()

# Ensure chembl_path is defined from previous steps
if 'chembl_path' not in locals():
    print("Warning: 'chembl_path' not found. Please run the ChEMBL downloader cell first.")
    # Fallback if chembl_path is not defined (e.g. if the previous cell was skipped)
    import chembl_downloader
    chembl_path = chembl_downloader.download_extract_sqlite(version="36")
    print(f"ChEMBL database path: {chembl_path}")

# 2. Query ChEMBL Database
if chembl_path:
    conn = sqlite3.connect(chembl_path)
    chembl_query = """
    SELECT
        md.chembl_id,
        cs.standard_inchi,
        cs.standard_inchi_key
    FROM
        molecule_dictionary md
    JOIN
        compound_structures cs ON md.molregno = cs.molregno
    """
    chembl_df = pd.read_sql_query(chembl_query, conn)
    conn.close()
    print("\nChEMBL data queried successfully.")
    print(f"Shape: {chembl_df.shape}")
    print("First 5 rows of ChEMBL data:")
    display(chembl_df.head())
else:
    print("Cannot query ChEMBL: chembl_path is not defined or invalid.")
    chembl_df = pd.DataFrame()

# 3. Merge DataFrames
if not npass_structure_df.empty and not chembl_df.empty:
    # Ensure column types are consistent for merging (e.g., stripping whitespace)
    npass_structure_df['InChI'] = npass_structure_df['InChI'].astype(str).str.strip()
    npass_structure_df['InChIKey'] = npass_structure_df['InChIKey'].astype(str).str.strip()
    chembl_df['standard_inchi'] = chembl_df['standard_inchi'].astype(str).str.strip()
    chembl_df['standard_inchi_key'] = chembl_df['standard_inchi_key'].astype(str).str.strip()

    # Merge on InChI and InChIKey (using standard_inchi and standard_inchi_key from ChEMBL for matching)
    merged_df = pd.merge(
        npass_structure_df,
        chembl_df,
        left_on=['InChI', 'InChIKey'],
        right_on=['standard_inchi', 'standard_inchi_key'],
        how='left' # Use left merge to keep all NPASS compounds and add ChEMBL info where available
    )

    # 4. Select Final Columns
    # The user asked for 'inchi' and 'inchikey' which are present in both, using NPASS's original here
    final_df = merged_df[[
        'np_id',
        'InChI', # NPASS original InChI
        'InChIKey', # NPASS original InChIKey
        'chembl_id',
        'standard_inchi',
        'standard_inchi_key'
    ]].copy()

    print("\nMerged DataFrame created successfully.")
    print(f"Shape: {final_df.shape}")
    print("First 5 rows of the new DataFrame with ChEMBL info:")
    display(final_df.head())
    print("\nValue counts for chembl_id (top 10):\n", final_df['chembl_id'].value_counts().head(10))
    print("\nNumber of NPASS compounds successfully mapped to ChEMBL IDs:", final_df['chembl_id'].nunique())
else:
    print("Merged DataFrame could not be created due to empty input DataFrames.")
    final_df = pd.DataFrame()

In [ ]:
import pandas as pd
import sqlite3

# 1. Load NPASS Natural Products Structure Data
try:
    # Corrected file path
    npass_structure_df = pd.read_csv(
        "/content/NPASS3.0_naturalproducts_structure.txt",
        sep="\t",
        header=0
    )
    print("NPASS Natural Products Structure data loaded successfully.")
    print(f"Shape: {npass_structure_df.shape}")
    print("First 5 rows of NPASS data:")
    display(npass_structure_df.head())
except FileNotFoundError:
    print("Error: NPASS3.0_naturalproducts_structure.txt not found. Please ensure it's in the correct path.")
    npass_structure_df = pd.DataFrame()

# Ensure chembl_path is defined from previous steps
if 'chembl_path' not in locals():
    print("Warning: 'chembl_path' not found. Please run the ChEMBL downloader cell first.")
    # Fallback if chembl_path is not defined (e.g. if the previous cell was skipped)
    import chembl_downloader
    chembl_path = chembl_downloader.download_extract_sqlite(version="36")
    print(f"ChEMBL database path: {chembl_path}")

# 2. Query ChEMBL Database
if chembl_path:
    conn = sqlite3.connect(chembl_path)
    chembl_query = """
    SELECT
        md.chembl_id,
        cs.standard_inchi,
        cs.standard_inchi_key
    FROM
        molecule_dictionary md
    JOIN
        compound_structures cs ON md.molregno = cs.molregno
    """
    chembl_df = pd.read_sql_query(chembl_query, conn)
    conn.close()
    print("\nChEMBL data queried successfully.")
    print(f"Shape: {chembl_df.shape}")
    print("First 5 rows of ChEMBL data:")
    display(chembl_df.head())
else:
    print("Cannot query ChEMBL: chembl_path is not defined or invalid.")
    chembl_df = pd.DataFrame()

# 3. Merge DataFrames
if not npass_structure_df.empty and not chembl_df.empty:
    # Ensure column types are consistent for merging (e.g., stripping whitespace)
    npass_structure_df['InChI'] = npass_structure_df['InChI'].astype(str).str.strip()
    npass_structure_df['InChIKey'] = npass_structure_df['InChIKey'].astype(str).str.strip()
    chembl_df['standard_inchi'] = chembl_df['standard_inchi'].astype(str).str.strip()
    chembl_df['standard_inchi_key'] = chembl_df['standard_inchi_key'].astype(str).str.strip()

    # Merge on InChI and InChIKey (using standard_inchi and standard_inchi_key from ChEMBL for matching)
    merged_df = pd.merge(
        npass_structure_df,
        chembl_df,
        left_on=['InChI', 'InChIKey'],
        right_on=['standard_inchi', 'standard_inchi_key'],
        how='left' # Use left merge to keep all NPASS compounds and add ChEMBL info where available
    )

    # 4. Select Final Columns
    # The user asked for 'inchi' and 'inchikey' which are present in both, using NPASS's original here
    final_df = merged_df[[
        'np_id',
        'InChI', # NPASS original InChI
        'InChIKey', # NPASS original InChIKey
        'chembl_id',
        'standard_inchi',
        'standard_inchi_key'
    ]].copy()

    print("\nMerged DataFrame created successfully.")
    print(f"Shape: {final_df.shape}")
    print("First 5 rows of the new DataFrame with ChEMBL info:")
    display(final_df.head())
    print("\nValue counts for chembl_id (top 10):\n", final_df['chembl_id'].value_counts().head(10))
    print("\nNumber of NPASS compounds successfully mapped to ChEMBL IDs:", final_df['chembl_id'].nunique())
else:
    print("Merged DataFrame could not be created due to empty input DataFrames.")
    final_df = pd.DataFrame()

In [ ]:
final_df.to_csv('natural_products_chembl_merged.csv', index=False)
print('File natural_products_chembl_merged.csv has been saved.')

### Loading `/content/NPASS3.0_species_info.txt`

species info เอาแค่ org_id, org_name , org_tax_id,species_tax_id species_name genus_tax_id, genus_name,genus tax id,family tax id ,fam tax name,kingdom name,kingdom id,super kingdom ,superkingdom id

In [ ]:
df_species_info = pd.read_csv('/content/NPASS3.0_species_info.txt', sep='\t')
display(df_species_info.head())

In [ ]:
df_species_info_selected = df_species_info[[
    'org_id', 'org_name', 'org_tax_id', 'species_tax_id', 'species_name',
    'genus_tax_id', 'genus_name', 'family_tax_id', 'family_name',
    'kingdom_name', 'kingdom_tax_id', 'superkingdom_name', 'superkingdom_tax_id'
]]
display(df_species_info_selected.head())

In [ ]:
df_species_info_selected.to_csv('species_info_selected.csv', index=False)
print('File species_info_selected.csv has been saved.')

### Loading `/content/NPASS3.0_target.txt`

target เอาเชื่อมฐานข้อมูล chembl แล้วเอาแค่ target id, targetname, target type ,chembl

In [ ]:
df_target = pd.read_csv('/content/NPASS3.0_target.txt', sep='\t')
display(df_target.head())

In [ ]:
df_target_chembl_info = df_target_merged_chembl[[
    'target_id',
    'target_name',
    'target_type',
    'chembl_id'
]].copy()

print("Shape of df_target_chembl_info:", df_target_chembl_info.shape)
display(df_target_chembl_info.head())

In [ ]:
df_target_chembl_info.to_csv('target_chembl_info.csv', index=False)
print('File target_chembl_info.csv has been saved.')

file activities เอา compond ชน target

In [ ]:
import pandas as pd

df_activities = pd.read_csv('/content/sample_data/NPASS3.0_activities.txt', sep='\t')
display(df_activities.head())

In [ ]:
df_activities_selected = df_activities[['np_id', 'target_id']]
display(df_activities_selected.head())

df_activities_selected.to_csv('activities_np_target_selected.csv', index=False)
print('File activities_np_target_selected.csv has been saved.')

# Task
Merge the `df_structure` and `df_generalinfo` DataFrames on the `np_id` column using an inner merge to ensure only matching natural products are included. From the resulting DataFrame, select the `np_id`, `InChI`, `InChIKey`, and `chembl_id` columns. Rename the `InChI` column to `standard_inchi` and the `InChIKey` column to `standard_inchi_key`. Finally, display the first 5 rows of this new DataFrame and save it as a CSV file named `natural_products_chembl_info.csv` without including the index.

## Merge DataFrames for ChEMBL IDs

### Subtask:
Merge `df_structure` and `df_generalinfo` on `np_id` to combine structural information with ChEMBL IDs. This will create a comprehensive table containing all the necessary information for the final output.


**Reasoning**:
To combine structural information with general information including ChEMBL IDs, I will merge `df_structure` and `df_generalinfo` using the `np_id` column as the key.



In [ ]:
df_merged_chembl = pd.merge(df_structure, df_generalinfo, on='np_id', how='inner')
display(df_merged_chembl.head())

**Reasoning**:
Now that the DataFrames are merged, I need to select the specified columns, rename them, display the head, and then save the resulting DataFrame to a CSV file as requested by the main task.



## Final Task

### Subtask:
Confirm that the task has been completed and the final DataFrame is ready for further use or inspection.


## Summary:

### Data Analysis Key Findings

*   `df_structure` and `df_generalinfo` DataFrames were successfully merged using an inner join on the `np_id` column, creating a combined DataFrame named `df_merged_chembl`.
*   A new DataFrame, `df_natural_products_chembl_info`, was created by selecting specific columns (`np_id`, `InChI`, `InChIKey`, and `chembl_id`) from `df_merged_chembl`.
*   The `InChI` column was successfully renamed to `standard_inchi`, and the `InChIKey` column was renamed to `standard_inchi_key` within `df_natural_products_chembl_info`.
*   The final DataFrame, `df_natural_products_chembl_info`, was saved as a CSV file named `natural_products_chembl_info.csv`, with the index excluded.

### Insights or Next Steps

*   The created `natural_products_chembl_info.csv` provides a consolidated and standardized dataset linking natural product identifiers to their InChI codes, InChIKeys, and ChEMBL IDs, ready for downstream analysis or database integration.
*   This dataset can now be used to query ChEMBL for detailed biological activity or target information related to these natural products, using the `chembl_id` column.


# Task
Install `chembl_downloader`, then load the NPASS natural products structure data from `/content/NPASS3.0_naturalproducts_structure.txt`, query ChEMBL for molecule dictionary information including InChI, InChIKey, and ChEMBL ID, and finally merge the NPASS structure data with the ChEMBL data to map NPASS compounds to ChEMBL IDs.

## Install chembl_downloader

### Subtask:
Install the missing `chembl_downloader` package.


**Reasoning**:
To install the missing `chembl_downloader` package, I will add the `!pip install chembl_downloader` command to the beginning of the specified code cell.



**Reasoning**:
The previous attempt to load `npass_structure_df` failed because of an incorrect file path. I need to correct the path from `/content/sample_data/NPASS3.0_naturalproducts_structure.txt` to `/content/NPASS3.0_naturalproducts_structure.txt`. The `chembl_downloader` is already installed, so I will remove the installation command.



## Summary:

### Data Analysis Key Findings

*   The `chembl_downloader` package was successfully installed.
*   NPASS Natural Products Structure data, containing 203,390 entries and 4 columns, was successfully loaded from `/content/NPASS3.0_naturalproducts_structure.txt`.
*   ChEMBL molecule dictionary information, including ChEMBL ID, InChI, and InChIKey, was queried, resulting in 2,854,815 entries and 3 columns.
*   The NPASS structure data was successfully merged with the ChEMBL data, creating a final DataFrame of 203,390 entries and 6 columns.
*   A total of 53,376 unique NPASS compounds were successfully mapped to ChEMBL IDs.

### Insights or Next Steps

*   The merging process successfully linked a significant portion of NPASS natural products to ChEMBL entries, enabling further analysis by leveraging the extensive bioactivity and chemical information available in ChEMBL.
*   For NPASS compounds not mapped to ChEMBL IDs, explore alternative methods for identification, such as substructure searching or cheminformatic tools, to maximize the number of integrated compounds.


# Task
Query ChEMBL for target information, selecting `chembl_id` and `uniprot_id` from the `target_dictionary` and `component_sequences` tables. Then, clean the `uniprot_id` column in `df_target` and merge it with the retrieved ChEMBL target data. Finally, select `target_id`, `target_name`, `target_type`, and `chembl_id`, display the first 5 rows, and save the result to `target_chembl_info.csv`.

## Query ChEMBL for Target Information

### Subtask:
Connect to the ChEMBL database and query for target `chembl_id` and `uniprot_id` by joining `target_dictionary` and `component_sequences` tables.


**Reasoning**:
To query the ChEMBL database for target information, I need to establish a connection, construct and execute a SQL query, and then display the results. I will use the `chembl_path` variable that was previously defined.



**Reasoning**:
The previous query failed because the `component_sequences` table does not contain the `uniprot_id` column, and the join condition was incorrect. The `uniprot_id` is available in the `target_components` table, which needs to be joined with `target_dictionary` on `tid` and `target_id` respectively to retrieve the `chembl_id` and `uniprot_id`.



**Reasoning**:
The previous queries failed because the `uniprot_id` column was not directly available in `target_components` or `component_sequences` tables as previously assumed. I need to join `target_dictionary` with `target_components` and then with `component_sequences` to correctly retrieve the UniProt accession, which is typically stored in the `accession` column of the `component_sequences` table, and alias it as `uniprot_id`.



**Reasoning**:
The previous query failed because the join condition between `target_dictionary` and `target_components` used an incorrect column name (`tc.target_id` instead of `tc.tid`). I will correct the join condition to `td.tid = tc.tid` to properly link the tables and retrieve the `chembl_id` and `uniprot_id`.



In [ ]:
import sqlite3

# Ensure chembl_path is defined (it should be from previous steps)
if 'chembl_path' not in locals():
    print("Warning: 'chembl_path' not found. Attempting to download ChEMBL again.")
    import chembl_downloader
    chembl_path = chembl_downloader.download_extract_sqlite(version="36")
    print(f"ChEMBL database path: {chembl_path}")

# 1. Establish a connection to the ChEMBL SQLite database
if chembl_path:
    conn = sqlite3.connect(chembl_path)

    # 2. Write a SQL query to get chembl_id and uniprot_id by joining target_dictionary, target_components, and component_sequences
    chembl_target_query = """
    SELECT
        td.chembl_id,
        cs.accession AS uniprot_id
    FROM
        target_dictionary td
    JOIN
        target_components tc ON td.tid = tc.tid
    JOIN
        component_sequences cs ON tc.component_id = cs.component_id
    WHERE
        cs.accession IS NOT NULL AND cs.accession != ''
    """

    # 3. Execute the SQL query and store results in a DataFrame
    chembl_targets_df = pd.read_sql_query(chembl_target_query, conn)

    # 4. Close the database connection
    conn.close()

    print("\nChEMBL target data queried successfully.")
    print(f"Shape: {chembl_targets_df.shape}")
    print("First 5 rows of ChEMBL target data:")
    display(chembl_targets_df.head())
else:
    print("Cannot query ChEMBL: chembl_path is not defined or invalid.")
    chembl_targets_df = pd.DataFrame()


In [ ]:
print(df_target['uniprot_id'].value_counts(dropna=False).head())
print(df_target['uniprot_id'].unique()[:10])

In [ ]:
import numpy as np

# Strip whitespace and replace all forms of 'n.a.' and 'nan' string with np.nan
df_target['uniprot_id'] = df_target['uniprot_id'].astype(str).str.strip()
df_target['uniprot_id'] = df_target['uniprot_id'].replace(['n.a.', 'n.a', 'nan'], np.nan)

print(df_target['uniprot_id'].value_counts(dropna=False).head())
print(df_target['uniprot_id'].unique()[:10])

In [ ]:
df_target_merged_chembl = pd.merge(
    df_target,
    chembl_targets_df,
    left_on='uniprot_id',
    right_on='uniprot_id',
    how='left'
)

print("Shape of df_target_merged_chembl:", df_target_merged_chembl.shape)
display(df_target_merged_chembl.head())

In [ ]:
df_target_chembl_info = df_target_merged_chembl[[
    'target_id',
    'target_name',
    'target_type',
    'chembl_id'
]].copy()

print("Shape of df_target_chembl_info:", df_target_chembl_info.shape)
display(df_target_chembl_info.head())

df_target_chembl_info.to_csv('target_chembl_info.csv', index=False)
print('File target_chembl_info.csv has been saved.')

In [ ]:
if 'chembl_targets_df' in locals() and not chembl_targets_df.empty:
    print('DataFrame chembl_targets_df exists and is not empty.')
    print('Shape of chembl_targets_df:', chembl_targets_df.shape)
    print('Columns of chembl_targets_df:', chembl_targets_df.columns.tolist())
    print('First 5 rows of chembl_targets_df:')
    display(chembl_targets_df.head())

    # Verify specific columns
    if 'chembl_id' in chembl_targets_df.columns and 'uniprot_id' in chembl_targets_df.columns:
        print("Both 'chembl_id' and 'uniprot_id' columns are present.")
    else:
        print("Warning: Expected columns 'chembl_id' or 'uniprot_id' are missing.")
else:
    print('DataFrame chembl_targets_df does not exist or is empty.')

In [ ]:
print('First 5 rows of df_target_chembl_info:')
display(df_target_chembl_info.head())

print('\nShape of df_target_chembl_info:', df_target_chembl_info.shape)

print('\nColumns of df_target_chembl_info:', df_target_chembl_info.columns.tolist())